In [1]:
# # Loan Default Prediction — Feature Engineering

# This notebook prepares the loan dataset for machine learning.

# The feature engineering process includes:
# - Loading the cleaned dataset
# - Converting the loan date into useful features
#- Removing identifier columns
#- Encoding categorical variables
#- Selecting the target variable
#- Preparing the feature matrix for model training
#- Checking the final dataset before model training

In [2]:
# Step 2 — Import libraries
import pandas as pd
import numpy as np

In [3]:
# Step 3 — Load the dataset
df = pd.read_csv("Loan_default.csv")

In [4]:
df.head()

,LoanID,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,Education,EmploymentType,MaritalStatus,HasMortgage,HasDependents,LoanPurpose,HasCoSigner,Default,Loan Date (DD/MM/YYYY)
0,I38PQUQS96,56,85994,50587,520,80,4,15.23,36,0.44,Bachelor's,Full-time,Divorced,Yes,Yes,Other,Yes,0,10/15/2018
1,HPSK72WA7R,69,50432,124440,458,15,1,4.81,60,0.68,Master's,Full-time,Married,No,No,Other,Yes,0,3/25/2016
2,C1OZ6DPJ8Y,46,84208,129188,451,26,3,21.17,24,0.31,Master's,Unemployed,Divorced,Yes,Yes,Auto,No,1,11/11/2013
3,V2KKSFM3UN,32,31713,44799,743,0,3,7.07,24,0.23,High School,Full-time,Married,No,No,Business,No,0,6/22/2017
4,EY08JDHTZP,60,20437,9139,633,8,4,6.51,48,0.73,Bachelor's,Unemployed,Divorced,No,Yes,Auto,No,0,6/9/2014


In [5]:
df.shape

(255347, 19)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255347 entries, 0 to 255346
Data columns (total 19 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   LoanID                  255347 non-null  object 
 1   Age                     255347 non-null  int64  
 2   Income                  255347 non-null  int64  
 3   LoanAmount              255347 non-null  int64  
 4   CreditScore             255347 non-null  int64  
 5   MonthsEmployed          255347 non-null  int64  
 6   NumCreditLines          255347 non-null  int64  
 7   InterestRate            255347 non-null  float64
 8   LoanTerm                255347 non-null  int64  
 9   DTIRatio                255347 non-null  float64
 10  Education               255347 non-null  object 
 11  EmploymentType          255347 non-null  object 
 12  MaritalStatus           255347 non-null  object 
 13  HasMortgage             255347 non-null  object 
 14  HasDependents       

In [7]:
df["LoanDate"] = pd.to_datetime(
    df["Loan Date (DD/MM/YYYY)"],
    format="%m/%d/%Y",
    errors="coerce"
)

In [10]:
df["LoanYear"] = df["LoanDate"].dt.year
df["LoanMonth"] = df["LoanDate"].dt.month

In [ ]:
df[[
    "Loan Date (DD/MM/YYYY)",
    "LoanDate",
    "LoanYear",
    "LoanMonth"
]].head()





LoanYear
2013    42785
2014    42122
2015    42521
2016    42705
2017    42377
2018    42837
Name: count, dtype: int64

In [12]:
df["LoanDate"].isna().sum()

np.int64(0)

In [14]:
df["LoanYear"].value_counts().sort_index()

LoanYear
2013    42785
2014    42122
2015    42521
2016    42705
2017    42377
2018    42837
Name: count, dtype: int64

In [15]:
X = df.drop(
    columns=[
        "Default",
        "LoanID",
        "Loan Date (DD/MM/YYYY)",
        "LoanDate"
    ]
)

y = df["Default"]

In [16]:
X.shape

(255347, 18)

In [17]:
X.columns

Index(['Age', 'Income', 'LoanAmount', 'CreditScore', 'MonthsEmployed',
       'NumCreditLines', 'InterestRate', 'LoanTerm', 'DTIRatio', 'Education',
       'EmploymentType', 'MaritalStatus', 'HasMortgage', 'HasDependents',
       'LoanPurpose', 'HasCoSigner', 'LoanYear', 'LoanMonth'],
      dtype='object')

In [18]:
categorical_columns = X.select_dtypes(
    include=["object"]
).columns.tolist()

categorical_columns

['Education',
 'EmploymentType',
 'MaritalStatus',
 'HasMortgage',
 'HasDependents',
 'LoanPurpose',
 'HasCoSigner']

In [19]:
numerical_columns = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

numerical_columns

['Age',
 'Income',
 'LoanAmount',
 'CreditScore',
 'MonthsEmployed',
 'NumCreditLines',
 'InterestRate',
 'LoanTerm',
 'DTIRatio']

In [20]:
# Step 10 — One-Hot Encoding

from sklearn.preprocessing import OneHotEncoder

In [21]:
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

In [22]:
encoded_data = encoder.fit_transform(
    X[categorical_columns]
)

In [23]:
encoded_data.shape

(255347, 22)

In [24]:
encoder.get_feature_names_out(categorical_columns)

array(["Education_Bachelor's", 'Education_High School',
       "Education_Master's", 'Education_PhD', 'EmploymentType_Full-time',
       'EmploymentType_Part-time', 'EmploymentType_Self-employed',
       'EmploymentType_Unemployed', 'MaritalStatus_Divorced',
       'MaritalStatus_Married', 'MaritalStatus_Single', 'HasMortgage_No',
       'HasMortgage_Yes', 'HasDependents_No', 'HasDependents_Yes',
       'LoanPurpose_Auto', 'LoanPurpose_Business',
       'LoanPurpose_Education', 'LoanPurpose_Home', 'LoanPurpose_Other',
       'HasCoSigner_No', 'HasCoSigner_Yes'], dtype=object)

In [25]:
numerical_data = X[numerical_columns + ["LoanYear", "LoanMonth"]]

In [26]:
numerical_data.shape

(255347, 11)

In [27]:
encoded_df = pd.DataFrame(
    encoded_data,
    columns=encoder.get_feature_names_out(categorical_columns),
    index=X.index
)

In [28]:
encoded_df.shape

(255347, 22)

In [29]:
X_final = pd.concat(
    [numerical_data, encoded_df],
    axis=1
)

In [30]:
X_final.shape

(255347, 33)

In [31]:
X_final.head()

,Age,Income,LoanAmount,CreditScore,MonthsEmployed,NumCreditLines,InterestRate,LoanTerm,DTIRatio,LoanYear,...,HasMortgage_Yes,HasDependents_No,HasDependents_Yes,LoanPurpose_Auto,LoanPurpose_Business,LoanPurpose_Education,LoanPurpose_Home,LoanPurpose_Other,HasCoSigner_No,HasCoSigner_Yes
0,56,85994,50587,520,80,4,15.23,36,0.44,2018,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,69,50432,124440,458,15,1,4.81,60,0.68,2016,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2,46,84208,129188,451,26,3,21.17,24,0.31,2013,...,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,32,31713,44799,743,0,3,7.07,24,0.23,2017,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
4,60,20437,9139,633,8,4,6.51,48,0.73,2014,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [33]:
X_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 255347 entries, 0 to 255346
Data columns (total 33 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Age                           255347 non-null  int64  
 1   Income                        255347 non-null  int64  
 2   LoanAmount                    255347 non-null  int64  
 3   CreditScore                   255347 non-null  int64  
 4   MonthsEmployed                255347 non-null  int64  
 5   NumCreditLines                255347 non-null  int64  
 6   InterestRate                  255347 non-null  float64
 7   LoanTerm                      255347 non-null  int64  
 8   DTIRatio                      255347 non-null  float64
 9   LoanYear                      255347 non-null  int32  
 10  LoanMonth                     255347 non-null  int32  
 11  Education_Bachelor's          255347 non-null  float64
 12  Education_High School         255347 non-nul

In [34]:
encoder.fit_transform(X[categorical_columns])

array([[1., 0., 0., ..., 1., 0., 1.],
       [0., 0., 1., ..., 1., 0., 1.],
       [0., 0., 1., ..., 0., 1., 0.],
       ...,
       [0., 1., 0., ..., 0., 0., 1.],
       [0., 1., 0., ..., 1., 1., 0.],
       [1., 0., 0., ..., 0., 0., 1.]])

In [35]:
## Step 16 — Train/Test Split

from sklearn.model_selection import train_test_split

In [36]:
# 2. Split the original features
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [37]:
X_train.shape, X_test.shape

((204277, 18), (51070, 18))

In [38]:
y_train.shape, y_test.shape

((204277,), (51070,))

In [40]:
# 4. Check the target distribution

y_train.value_counts(normalize=True) * 100

Default
0    88.387337
1    11.612663
Name: proportion, dtype: float64

In [41]:
y_test.value_counts(normalize=True) * 100

Default
0    88.386528
1    11.613472
Name: proportion, dtype: float64

In [42]:
## Step 17 — Build the preprocessing pipeline
# Numerical → keep / optionally scale
# Categorical → One-Hot Encode

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [43]:
# 2. Define our columns
categorical_columns

['Education',
 'EmploymentType',
 'MaritalStatus',
 'HasMortgage',
 'HasDependents',
 'LoanPurpose',
 'HasCoSigner']

In [44]:
numerical_columns

['Age',
 'Income',
 'LoanAmount',
 'CreditScore',
 'MonthsEmployed',
 'NumCreditLines',
 'InterestRate',
 'LoanTerm',
 'DTIRatio']

In [45]:
numerical_features = numerical_columns + ["LoanYear", "LoanMonth"]

In [46]:
numerical_features

['Age',
 'Income',
 'LoanAmount',
 'CreditScore',
 'MonthsEmployed',
 'NumCreditLines',
 'InterestRate',
 'LoanTerm',
 'DTIRatio',
 'LoanYear',
 'LoanMonth']

In [57]:
## What this does
#   For numerical features: → keep them as numerical values
#   For categorical features: → One-Hot Encode them
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_columns
        )
    ]
)

In [58]:
## Step 18 — Fit ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)

In [59]:
# transform  test data.
X_test_processed = preprocessor.transform(X_test)

In [60]:
X_train_processed.shape, X_test_processed.shape

((204277, 33), (51070, 33))

In [61]:
## Step 19 — Save the feature names
# Before we train models, let's make the processed data easier to understand.

feature_names = preprocessor.get_feature_names_out()

len(feature_names)

33

In [62]:
feature_names

array(['num__Age', 'num__Income', 'num__LoanAmount', 'num__CreditScore',
       'num__MonthsEmployed', 'num__NumCreditLines', 'num__InterestRate',
       'num__LoanTerm', 'num__DTIRatio', 'num__LoanYear',
       'num__LoanMonth', "cat__Education_Bachelor's",
       'cat__Education_High School', "cat__Education_Master's",
       'cat__Education_PhD', 'cat__EmploymentType_Full-time',
       'cat__EmploymentType_Part-time',
       'cat__EmploymentType_Self-employed',
       'cat__EmploymentType_Unemployed', 'cat__MaritalStatus_Divorced',
       'cat__MaritalStatus_Married', 'cat__MaritalStatus_Single',
       'cat__HasMortgage_No', 'cat__HasMortgage_Yes',
       'cat__HasDependents_No', 'cat__HasDependents_Yes',
       'cat__LoanPurpose_Auto', 'cat__LoanPurpose_Business',
       'cat__LoanPurpose_Education', 'cat__LoanPurpose_Home',
       'cat__LoanPurpose_Other', 'cat__HasCoSigner_No',
       'cat__HasCoSigner_Yes'], dtype=object)

In [63]:
y_train.value_counts()

Default
0    180555
1     23722
Name: count, dtype: int64

In [64]:
y_test.value_counts()

Default
0    45139
1     5931
Name: count, dtype: int64

In [65]:
import joblib

joblib.dump(
    preprocessor,
    "preprocessor.pkl"
)

joblib.dump(
    X_train_processed,
    "X_train_processed.pkl"
)

joblib.dump(
    X_test_processed,
    "X_test_processed.pkl"
)

joblib.dump(
    y_train,
    "y_train.pkl"
)

joblib.dump(
    y_test,
    "y_test.pkl"
)

['y_test.pkl']